# Benchmark de NBody no Raspberry Pi 5

Autor: Raphael Figueiredo Secchin

Data: 21/09/2026

---
## <font color="#1C77C3" >Informações de Sistema</font>

Nome do sistema operacional, nome do computador na rede, versão do kernel, arquitetura de hardware (32/64 bits), etc :

In [1]:
!uname -a

Linux raspberrypi 6.18.34+rpt-rpi-2712 #1 SMP PREEMPT Debian 1:6.18.34-1+rpt1 (2026-06-09) aarch64 GNU/Linux


Nome e versão do sistema operacional :

In [2]:
!lsb_release -a

No LSB modules are available.
Distributor ID:	Debian
Description:	Debian GNU/Linux 13 (trixie)
Release:	13
Codename:	trixie


Diversas informações da CPU, como nome do processador, frequência em MHz da CPU, número de núcleos/cores, número de threads, memória cache, arquitetura de hardware (32/64 bits), etc :

In [3]:
!lscpu

Architecture:                aarch64
  CPU op-mode(s):            32-bit, 64-bit
  Byte Order:                Little Endian
CPU(s):                      4
  On-line CPU(s) list:       0-3
Vendor ID:                   ARM
  Model name:                Cortex-A76
    Model:                   1
    Thread(s) per core:      1
    Core(s) per cluster:     4
    Socket(s):               -
    Cluster(s):              1
    Stepping:                r4p1
    Frequency boost:         disabled
    CPU(s) scaling MHz:      100%
    CPU max MHz:             2400,0000
    CPU min MHz:             1500,0000
    BogoMIPS:                108,00
    Flags:                   fp asimd evtstrm aes pmull sha1 sha2 crc32 atomics 
                             fphp asimdhp cpuid asimdrdm lrcpc dcpop asimddp
Caches (sum of all):         
  L1d:                       256 KiB (4 instances)
  L1i:                       256 KiB (4 instances)
  L2:                        2 MiB (4 instances)
  L3:                    

Partições do sistema de arquivos do sistema operacional :

In [4]:
!df -h

Sist. Arq.      Tam. Usado Disp. Uso% Montado em
udev            3,9G     0  3,9G   0% /dev
tmpfs           1,6G   14M  1,6G   1% /run
/dev/mmcblk0p2  117G  8,9G  104G   8% /
tmpfs           4,0G  624K  4,0G   1% /dev/shm
tmpfs           5,0M   48K  5,0M   1% /run/lock
tmpfs           1,0M     0  1,0M   0% /run/credentials/systemd-journald.service
tmpfs           4,0G   16K  4,0G   1% /tmp
/dev/mmcblk0p1  505M   87M  418M  18% /boot/firmware
tmpfs           806M  272K  806M   1% /run/user/1000
tmpfs           1,0M     0  1,0M   0% /run/credentials/getty@tty1.service
tmpfs           1,0M     0  1,0M   0% /run/credentials/serial-getty@ttyAMA10.service
/dev/sda2       954G  310G  645G  33% /media/icnbody/Pendrive1TB


Memória RAM total e em uso pelo sistema operacional :

In [5]:
!free

               total       usada       livre    compart.  buff/cache  disponível
Mem.:        8251776     1776496     5557952      419568     1287248     6475280
Swap:        2097136           0     2097136


Versão do compilador C/C++ gcc :

In [6]:
!gcc --version

gcc (Debian 14.2.0-19) 14.2.0
Copyright (C) 2024 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



O módulo Pyton "platform" fornece diversas informações do sistema (computador, sistema operacional, Python, etc).

In [7]:
import platform

In [8]:
platform.platform()

'Linux-6.18.34+rpt-rpi-2712-aarch64-with-glibc2.41'

Mais detalhes, como nome do computador na rede (node), arquitetura de hardware (32/64 bits), etc:

In [9]:
platform.uname()

uname_result(system='Linux', node='raspberrypi', release='6.18.34+rpt-rpi-2712', version='#1 SMP PREEMPT Debian 1:6.18.34-1+rpt1 (2026-06-09)', machine='aarch64')

### Informações sobre Python e módulos

#### Python

Número da versão de Python:

In [10]:
platform.python_version()

'3.14.6'

data da versão:

In [11]:
platform.python_build()

('main', 'Aug 11 2026 10:24:17')

compilador C/C++ utilizado para criar tal versão de Python :

In [12]:
platform.python_compiler()

'GCC 14.4.0'

In [15]:
import numba as nb

In [16]:
nb.__version__

'0.67.0'

In [17]:
import numpy as np

In [18]:
np.__version__

'2.5.2'

### Informações sobre GPU

Mostra versão do driver NVidia, versão de CUDA e várias informações da GPU : nome, temperatura, potência usada e máxima, RAM usada e máxima, etc:

In [19]:
!nvidia-smi

/bin/bash: linha 1: nvidia-smi: comando não encontrado


Mostra dados do compilador CUDA, como versão:

In [20]:
!nvcc --version

/bin/bash: linha 1: nvcc: comando não encontrado


### Via Numba

In [21]:
from numba import cuda

Testa se CUDA está disponível, i. e., se tem GPU e se software CUDA foi instalado :

In [22]:
cuda.is_available()

False

Mostra identificação (começa de zero) da GPU, nome da GPU, CC (Compute Capability), etc :

In [23]:
cuda.detect()

CudaSupportError: Error at driver init: 

CUDA driver library cannot be found.
If you are sure that a CUDA driver is installed,
try setting environment variable NUMBA_CUDA_DRIVER
with the file path of the CUDA driver shared library.
:

Mostra RAM livre e total da GPU com identificação 0 (zero), em bytes :

In [24]:
cuda.current_context(0).get_memory_info()

CudaSupportError: Error at driver init: 

CUDA driver library cannot be found.
If you are sure that a CUDA driver is installed,
try setting environment variable NUMBA_CUDA_DRIVER
with the file path of the CUDA driver shared library.
:

---
## <font color="#5EAAE8">Inicialização do Problema</font>

In [25]:
from numba import njit, prange, cuda
import numpy as np
import math

In [26]:
espacamento = 10
passos = 10
g = 1
epsilon = 5
dt = 0.1
threadsperblock = 32
numCorpos = 32**3
corposPorEixo3D = int(math.cbrt(numCorpos))

A função calculaForcas calcula o valor das forças do sistema utilizando a fórmula:

$$ F_i = ∑_{\substack{j = 1 \\ j \neq i}}^{N} \frac{G m_{i} m_{j}}{|\vec{r}_{ij}|^{2}}$$

A função calculaEnergiaK calcula o valor da energia cinética utilizando a fórmula:

$$ K = ∑^{N}_{i = 1}\frac{m_{i}|\vec{v}_{i}|^{2}}{2}$$

A função calculaEnergiaU calcula o valor da energia cinética utilizando a fórmula:

$$ U = ∑^{N}_{i = 1}∑^{N}_{\substack{j = 1 \\ j \neq i}}\frac{G m_{i} m_{j}}{|r_{ij}|}$$

---
## <font color="#33AAFF">Python Puro</font>

---
### <font color="#E86C4A">3D</font>

In [27]:
x = []
y = []
z = []
vx = [1.0] * numCorpos
vy = [1.0] * numCorpos
vz = [1.0] * numCorpos
m = [1.0] * numCorpos
xTemp = 0
yTemp = 0
zTemp = 0

forcaX = [0.0] * numCorpos
forcaY = [0.0] * numCorpos
forcaZ = [0.0] * numCorpos

for i in range(corposPorEixo3D):
  xTemp += espacamento
  yTemp = 0
  for j in range(corposPorEixo3D):
    yTemp += espacamento
    zTemp = 0
    for k in range(corposPorEixo3D):
      zTemp += espacamento
      x.append(xTemp)
      y.append(yTemp)
      z.append(zTemp)

In [28]:
def calculaForcas(numCorpos, x, y, z, m, g, epsilon, forcaX, forcaY, forcaZ):
  for i in range(numCorpos):
    for j in range(i + 1, numCorpos):
      dx = x[j] - x[i]
      dy = y[j] - y[i]
      dz = z[j] - z[i]

      dist = (dx*dx + dy*dy + dz*dz + epsilon*epsilon)
      invDist = dist**(-0.5)
      invDist = invDist * invDist * invDist

      forcaAtual = (g * m[i] * m[j]) * invDist

      forcaX[i] += forcaAtual * dx
      forcaY[i] += forcaAtual * dy
      forcaZ[i] += forcaAtual * dz

      forcaX[j] += -forcaAtual * dx
      forcaY[j] += -forcaAtual * dy
      forcaZ[j] += -forcaAtual * dz

In [29]:
def movimentaCorpo(dt, vx, vy, vz, x, y, z, m, numCorpos, g, epsilon, forcaX, forcaY, forcaZ):
  for k in range(numCorpos):
    forcaX[k] = 0.0
    forcaY[k] = 0.0
    forcaZ[k] = 0.0

  calculaForcas(numCorpos, x, y, z, m, g, epsilon, forcaX, forcaY, forcaZ)
  for j in range(numCorpos):

    acelX = forcaX[j] / m[j]
    acelY = forcaY[j] / m[j]
    acelZ = forcaZ[j] / m[j]

    vx[j] = vx[j] + acelX * dt
    vy[j] = vy[j] + acelY * dt
    vz[j] = vz[j] + acelZ * dt

    x[j] = x[j] + vx[j] * dt
    y[j] = y[j] + vy[j] * dt
    z[j] = z[j] + vz[j] * dt

Tempo de Execução

In [30]:
%%time
for i in range(passos):
  movimentaCorpo(dt, vx, vy, vz, x, y, z, m, numCorpos, g, epsilon, forcaX, forcaY, forcaZ)

CPU times: user 1h 26min 27s, sys: 182 ms, total: 1h 26min 27s
Wall time: 1h 26min 28s


---
## <font color="#33AAFF">Numba CPU</font>

---
### <font color="#E86C4A">3D</font>

In [31]:
x = []
y = []
z = []
vx = [1.0] * numCorpos
vy = [1.0] * numCorpos
vz = [1.0] * numCorpos
m = [1.0] * numCorpos
xTemp = 0
yTemp = 0
zTemp = 0

forcaX = [0.0] * numCorpos
forcaY = [0.0] * numCorpos
forcaZ = [0.0] * numCorpos

for i in range(corposPorEixo3D):
  xTemp += espacamento
  yTemp = 0
  for j in range(corposPorEixo3D):
    yTemp += espacamento
    zTemp = 0
    for k in range(corposPorEixo3D):
      zTemp += espacamento
      x.append(xTemp)
      y.append(yTemp)
      z.append(zTemp)

In [32]:
x = np.array(x, dtype=np.float64)
y = np.array(y, dtype=np.float64)
z = np.array(z, dtype=np.float64)
m = np.array(m, dtype=np.float64)
vx = np.array(vx, dtype=np.float64)
vy = np.array(vy, dtype=np.float64)
vz = np.array(vz, dtype=np.float64)

forcaX = np.array(forcaX, dtype=np.float64)
forcaY = np.array(forcaY, dtype=np.float64)
forcaZ = np.array(forcaZ, dtype=np.float64)

In [33]:
@njit(cache=True)
def calculaForcas(numCorpos, x, y, z, m, g, epsilon, forcaX, forcaY, forcaZ):
  for i in range(numCorpos):
    for j in range(i + 1, numCorpos):
      dx = x[j] - x[i]
      dy = y[j] - y[i]
      dz = z[j] - z[i]

      dist = (dx*dx + dy*dy + dz*dz + epsilon*epsilon)
      invDist = dist**(-0.5)
      invDist = invDist * invDist * invDist

      forcaAtual = (g * m[i] * m[j]) * invDist

      forcaX[i] += forcaAtual * dx
      forcaY[i] += forcaAtual * dy
      forcaZ[i] += forcaAtual * dz

      forcaX[j] += -forcaAtual * dx
      forcaY[j] += -forcaAtual * dy
      forcaZ[j] += -forcaAtual * dz

In [34]:
@njit(cache=True)
def movimentaCorpo(dt, vx, vy, vz, x, y, z, m, numCorpos, g, epsilon, forcaX, forcaY, forcaZ):
    for k in range(numCorpos):
      forcaX[k] = 0.0
      forcaY[k] = 0.0
      forcaZ[k] = 0.0

    calculaForcas(numCorpos, x, y, z, m, g, epsilon, forcaX, forcaY, forcaZ)
    for j in range(numCorpos):

      acelX = forcaX[j] / m[j]
      acelY = forcaY[j] / m[j]
      acelZ = forcaZ[j] / m[j]

      vx[j] = vx[j] + acelX * dt
      vy[j] = vy[j] + acelY * dt
      vz[j] = vz[j] + acelZ * dt

      x[j] = x[j] + vx[j] * dt
      y[j] = y[j] + vy[j] * dt
      z[j] = z[j] + vz[j] * dt

In [35]:
%%time
for i in range(passos):
  movimentaCorpo(dt, vx, vy, vz, x, y, z, m, numCorpos, g, epsilon, forcaX, forcaY, forcaZ)

CPU times: user 3min 41s, sys: 268 ms, total: 3min 41s
Wall time: 3min 42s


---
## <font color="#33AAFF">Numba CPU //</font>

---
### <font color="#E86C4A">3D</font>

In [36]:
x = []
y = []
z = []
vx = [1.0] * numCorpos
vy = [1.0] * numCorpos
vz = [1.0] * numCorpos
m = [1.0] * numCorpos
xTemp = 0
yTemp = 0
zTemp = 0

forcaX = [0.0] * numCorpos
forcaY = [0.0] * numCorpos
forcaZ = [0.0] * numCorpos

for i in range(corposPorEixo3D):
  xTemp += espacamento
  yTemp = 0
  for j in range(corposPorEixo3D):
    yTemp += espacamento
    zTemp = 0
    for k in range(corposPorEixo3D):
      zTemp += espacamento
      x.append(xTemp)
      y.append(yTemp)
      z.append(zTemp)

In [37]:
x = np.array(x, dtype=np.float64)
y = np.array(y, dtype=np.float64)
z = np.array(z, dtype=np.float64)
m = np.array(m, dtype=np.float64)
vx = np.array(vx, dtype=np.float64)
vy = np.array(vy, dtype=np.float64)
vz = np.array(vz, dtype=np.float64)

forcaX = np.array(forcaX, dtype=np.float64)
forcaY = np.array(forcaY, dtype=np.float64)
forcaZ = np.array(forcaZ, dtype=np.float64)

In [38]:
@njit(parallel=True, cache=True)
def calculaForcas(numCorpos, x, y, z, m, g, epsilon, forcaX, forcaY, forcaZ):
  for i in prange(numCorpos):
    for j in range(numCorpos):
      dx = x[j] - x[i]
      dy = y[j] - y[i]
      dz = z[j] - z[i]

      dist = (dx*dx + dy*dy + dz*dz + epsilon*epsilon)
      invDist = dist**(-0.5)
      invDist = invDist * invDist * invDist

      forcaAtual = (g * m[i] * m[j]) * invDist

      forcaX[i] += forcaAtual * dx
      forcaY[i] += forcaAtual * dy
      forcaZ[i] += forcaAtual * dz

In [39]:
@njit(parallel=True, cache=True)
def movimentaCorpo(dt, vx, vy, vz, x, y, z, m, numCorpos, g, epsilon, forcaX, forcaY, forcaZ):
  for k in prange(numCorpos):
    forcaX[k] = 0.0
    forcaY[k] = 0.0
    forcaZ[k] = 0.0

  calculaForcas(numCorpos, x, y, z, m, g, epsilon, forcaX, forcaY, forcaZ)
  for j in prange(numCorpos):

    acelX = forcaX[j] / m[j]
    acelY = forcaY[j] / m[j]
    acelZ = forcaZ[j] / m[j]

    vx[j] = vx[j] + acelX * dt
    vy[j] = vy[j] + acelY * dt
    vz[j] = vz[j] + acelZ * dt

    x[j] = x[j] + vx[j] * dt
    y[j] = y[j] + vy[j] * dt
    z[j] = z[j] + vz[j] * dt

In [40]:
%%time
for i in range(passos):
  movimentaCorpo(dt, vx, vy, vz, x, y, z, m, numCorpos, g, epsilon, forcaX, forcaY, forcaZ)

CPU times: user 5min 20s, sys: 155 ms, total: 5min 20s
Wall time: 1min 26s


---
## <font color="#33AAFF">Numba GPU</font>

In [ ]:
blockspergrid = (numCorpos + (threadsperblock - 1)) // threadsperblock

---
### <font color="#E86C4A">3D</font>

In [ ]:
xH = []
yH = []
zH = []
vxH = [1.0] * numCorpos
vyH = [1.0] * numCorpos
vzH = [1.0] * numCorpos
mH = [1.0] * numCorpos
kH = [0.0] * numCorpos
uH = [0.0] * numCorpos
xTemp = 0

forcaXH = [0.0] * numCorpos
forcaYH = [0.0] * numCorpos
forcaZH = [0.0] * numCorpos

for i in range(corposPorEixo3D):
  xTemp += espacamento
  yTemp = 0
  for j in range(corposPorEixo3D):
    yTemp += espacamento
    zTemp = 0
    for k in range(corposPorEixo3D):
      zTemp += espacamento
      xH.append(xTemp)
      yH.append(yTemp)
      zH.append(zTemp)

xH = np.array(xH, dtype=np.float64)
yH = np.array(yH, dtype=np.float64)
zH = np.array(zH, dtype=np.float64)
mH = np.array(mH, dtype=np.float64)
vxH = np.array(vxH, dtype=np.float64)
vyH = np.array(vyH, dtype=np.float64)
vzH = np.array(vzH, dtype=np.float64)

forcaXH = np.array(forcaXH, dtype=np.float64)
forcaYH = np.array(forcaYH, dtype=np.float64)
forcaZH = np.array(forcaZH, dtype=np.float64)
kH = np.array(kH, dtype=np.float64)
uH = np.array(uH, dtype=np.float64)

xD = cuda.to_device(xH)
yD = cuda.to_device(yH)
zD = cuda.to_device(zH)
vxD = cuda.to_device(vxH)
vyD = cuda.to_device(vyH)
vzD = cuda.to_device(vzH)
mD = cuda.to_device(mH)
kD = cuda.to_device(kH)
uD = cuda.to_device(uH)
forcaXD = cuda.to_device(forcaXH)
forcaYD = cuda.to_device(forcaYH)
forcaZD = cuda.to_device(forcaZH)

In [ ]:
@cuda.jit(device=True)
def calculaForcas(numCorpos, xD, yD, zD, mD, g, epsilon, forcaXD, forcaYD, forcaZD, i):
  xLocal = cuda.shared.array(threadsperblock, cuda.float64)
  yLocal = cuda.shared.array(threadsperblock, cuda.float64)
  zLocal = cuda.shared.array(threadsperblock, cuda.float64)
  mLocal = cuda.shared.array(threadsperblock, cuda.float64)

  xDAtual = xD[i]
  yDAtual = yD[i]
  zDAtual = zD[i]
  mDAtual = mD[i]

  lx = cuda.threadIdx.x

  forcaLocalX = 0.0
  forcaLocalY = 0.0
  forcaLocalZ = 0.0

  ep2 = epsilon*epsilon
  for j in range(numCorpos // threadsperblock):
    xLocal[lx] = xD[lx + (threadsperblock * j)]
    yLocal[lx] = yD[lx + (threadsperblock * j)]
    zLocal[lx] = zD[lx + (threadsperblock * j)]
    mLocal[lx] = mD[lx + (threadsperblock * j)]

    cuda.syncthreads()
    for k in range(threadsperblock):
      dx = xLocal[k] - xDAtual
      dy = yLocal[k] - yDAtual
      dz = zLocal[k] - zDAtual
      dist = (dx*dx + dy*dy + dz*dz + ep2)

      invDist = dist**(-0.5)
      invDist = invDist * invDist * invDist

      forcaAtual = (g * mDAtual * mLocal[k]) * invDist
      forcaLocalX += forcaAtual * dx
      forcaLocalY += forcaAtual * dy
      forcaLocalZ += forcaAtual * dz

    cuda.syncthreads()

  forcaXD[i] = forcaLocalX
  forcaYD[i] = forcaLocalY
  forcaZD[i] = forcaLocalZ

In [ ]:
@cuda.jit
def movimentaCorpo(dt, vxD, vyD, vzD, xD, yD, zD, mD, numCorpos, g, epsilon, forcaXD, forcaYD, forcaZD):
    i = cuda.grid(1)

    if(i < numCorpos):
      forcaXD[i] = 0.0
      forcaYD[i] = 0.0
      forcaZD[i] = 0.0
      calculaForcas(numCorpos, xD, yD, zD, mD, g, epsilon, forcaXD, forcaYD, forcaZD, i)

      acelX = forcaXD[i] / mD[i]
      acelY = forcaYD[i] / mD[i]
      acelZ = forcaZD[i] / mD[i]

      vxD[i] = vxD[i] + acelX * dt
      vyD[i] = vyD[i] + acelY * dt
      vzD[i] = vzD[i] + acelZ * dt

      xD[i] = xD[i] + vxD[i] * dt
      yD[i] = yD[i] + vyD[i] * dt
      zD[i] = zD[i] + vzD[i] * dt

In [ ]:
%%time
for i in range(passos):
  movimentaCorpo[blockspergrid, threadsperblock](dt, vxD, vyD, vzD, xD, yD, zD, mD, numCorpos, g, epsilon, forcaXD, forcaYD, forcaZD)